# 13장 생성형 AI API

이 노트북은 웹 교재 [13장 생성형 AI API](https://pidocs.aifusionlab.org/ai-python/ch13/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 13.1 Gemini API 호출

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch13/13-1/)

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = os.environ.get("GEMINI_API_KEY")     # PC에서는 환경변수 사용

MODEL = "gemini-2.5-flash"                      # 사용 가능한 모델 이름은 공식 문서에서 확인
print("API 키 준비:", key is not None)

In [ ]:
if key:
    from google import genai
    client = genai.Client(api_key=key)
    response = client.models.generate_content(
        model=MODEL,
        contents="Wi-Fi 신호 세기 단위 dBm의 의미를 고등학생에게 두 문장으로 설명하시오.")
    print(response.text)
    print(response.usage_metadata)
else:
    print("API 키가 없어 호출을 건너뜁니다.")

## 13.2 프롬프트 설계

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch13/13-2/)

In [ ]:
def make_prompt(summary_text):
    role = "당신은 정보통신학과 실습 보고서를 작성하는 조교입니다. 독자는 같은 수업의 학생입니다."
    rules = ("- 3문장으로 작성합니다.\n"
             "- 아래 [측정 요약]에 있는 수치만 사용하고, 새로운 수치를 계산하거나 추측하지 않습니다.\n"
             "- 신호 세기는 dBm 단위를 붙여 씁니다.\n"
             "- 가장 강한 지점과 가장 약한 지점을 반드시 언급합니다.")
    return f"{role}\n\n[작성 규칙]\n{rules}\n\n[측정 요약]\n{summary_text}\n"

print(make_prompt("D1: 중앙값 -52 dBm\nD2: 중앙값 -66 dBm"))

In [ ]:
EXAMPLE = ("[예시]\n"
           "조도 측정 결과 창가가 중앙값 82로 가장 밝고, 복도 쪽이 중앙값 35로 가장 어두웠습니다. "
           "두 지점의 차이는 47입니다. 복도 쪽 자리는 조명 보강이 필요합니다.\n")

def make_prompt_fewshot(summary_text):
    return make_prompt(summary_text) + "\n" + EXAMPLE

if key:
    r = client.models.generate_content(model=MODEL, contents=make_prompt_fewshot(
        "D1: 중앙값 -52 dBm\nD2: 중앙값 -66 dBm\nD3: 중앙값 -62 dBm\nD4: 중앙값 -65.5 dBm"))
    print(r.text)
else:
    print("API 키가 없어 호출을 건너뜁니다.")

## 13.3 수치 계산과 문장 생성의 분리

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch13/13-3/)

In [ ]:
import pandas as pd

rssi = pd.read_csv("class_rssi_20261104.csv")
g = rssi.groupby("zone")["rssi"]
summary = pd.DataFrame({
    "count": g.size(),
    "median": g.median(),
    "weak_pct": g.apply(lambda v: round((v <= -75).mean() * 100, 1)),
})
summary

In [ ]:
def summary_to_text(s):
    lines = []
    for zone, row in s.iterrows():
        lines.append(f"{zone}: 기록 {int(row['count'])}건, 중앙값 {row['median']:g} dBm, "
                     f"-75 dBm 이하 비율 {row['weak_pct']:g}%")
    return "\n".join(lines)

summary_text = summary_to_text(summary)
print(summary_text)

In [ ]:
if key:
    report = client.models.generate_content(model=MODEL, contents=make_prompt(summary_text)).text
    print(report)
else:
    report = None
    print("API 키가 없어 호출을 건너뜁니다.")

## 13.4 출력 검증

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch13/13-4/)

In [ ]:
import re

def extract_numbers(text):
    """문장에서 숫자를 모두 찾아 실수 리스트로 반환한다."""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", text)]

print(extract_numbers("D1이 중앙값 -52 dBm으로 가장 강하고, D2는 -66 dBm입니다."))

In [ ]:
def allowed_values(s):
    vals = set()
    for _, row in s.iterrows():
        vals.update([float(row["count"]), float(row["median"]), float(row["weak_pct"])])
    vals.update([1.0, 2.0, 3.0, 4.0, -75.0])     # 지점 번호, 음영 기준
    return vals

def check_report(text, s):
    """요약값·허용 목록에 없는 수치를 리스트로 반환한다. 비어 있으면 통과."""
    ok = allowed_values(s)
    return [x for x in extract_numbers(text) if x not in ok]

test_text = ("D1이 중앙값 -52 dBm으로 가장 강하고, D2가 -66 dBm으로 가장 약했습니다. "
             "D2는 -75 dBm 이하 기록이 21.5%였고, D3는 -58 dBm이었습니다.")
print("검증 결과:", check_report(test_text, summary))

In [ ]:
def safe_report(s, max_try=3):
    text_in = summary_to_text(s)
    if not key:
        return "API 키가 없어 요약 표로 보고합니다.\n" + text_in
    for i in range(max_try):
        text = client.models.generate_content(model=MODEL, contents=make_prompt(text_in)).text
        bad = check_report(text, s)
        if not bad:
            return text
        print(f"{i + 1}회 검증 실패: {bad}")
    return "검증을 통과한 문장이 없어 요약 표로 보고합니다.\n" + text_in

print(safe_report(summary))